# Classificação do Fashion-MNIST usando Convolutional Neural Networks (CNN)

Este notebook implementa uma Rede Neural Convolucional (CNN) para resolver o mesmo problema abordado no notebook [anterior](01_fashion_mnist_ml.ipynb) — a classificação de imagens do Fashion-MNIST.

Como as CNNs são especialmente projetadas para lidar com dados visuais, esta abordagem permite uma comparação direta entre o desempenho de uma arquitetura convolucional e uma arquitetura totalmente densa (MLP), destacando como o uso de convoluções melhora a extração de padrões espaciais nas imagens.

O objetivo é mostrar, na prática, como CNNs tendem a alcançar maior acurácia e generalização em tarefas de visão computacional, mesmo em datasets simples como o Fashion-MNIST.

## Por que CNNs são superiores para imagens?

### Preservação da Estrutura Espacial
- Redes densas: flatten -> perde relação entre pixels vizinhos
- CNNs: mantêm a estrutura 2D -> aprendem padrões locais

### Compartilhamento de Pesos (Weight Sharing)
- Mesmos filtros aplicados em toda a imagem
- Drasticamente reduz número de parâmetros
- Aprende features invariantes à posição

### Hierarquia de Features
- Primeiras camadas: bordas, linhas, texturas básicas
- Camadas intermediárias: padrões, formas
- Últimas camadas: objetos completos, conceitos abstratos

## 0. Importação de Bibliotecas

In [1]:
import time
import numpy as np

from IPython.display import clear_output

import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Dense,
    Conv2D,
    MaxPool2D,
    Flatten,
    Dropout,
    BatchNormalization,
)
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras import mixed_precision

# Devido a warnings de "Allocation exceeds 10% of free system memory" vamos ativar o FP16
# RTX 3070 (Ampere) tem Tensor Cores que aceleram mixed precision (float16)
# Pesos -> Ficam em float32 (para estabilidade)
# Ativações e cálculos interno -> Em float16 nos Tensor Cores (mais rápido)
# Saída -> Também em float16
mixed_precision.set_global_policy("mixed_float16")

# Reprodutibilidade
np.random.seed(42)

2025-11-16 15:15:05.912593: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Funções e callbacks

In [2]:
class PlotLearning(keras.callbacks.Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.val_losses = []
        self.acc = []
        self.val_acc = []
        self.epochs = []

    def on_epoch_end(self, epoch, logs={}):
        self.epochs.append(epoch + 1)
        self.losses.append(logs.get("loss"))
        self.val_losses.append(logs.get("val_loss"))
        self.acc.append(logs.get("accuracy"))
        self.val_acc.append(logs.get("val_accuracy"))

        clear_output(wait=True)

        fig = make_subplots(
            rows=1,
            cols=2,
            subplot_titles=("Loss", "Accuracy"),
            horizontal_spacing=0.12,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.losses,
                mode="lines+markers",
                name="train_loss",
                line=dict(color="#1f77b4"),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.val_losses,
                mode="lines+markers",
                name="val_loss",
                line=dict(color="#d62728"),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.acc,
                mode="lines+markers",
                name="train_acc",
                line=dict(color="#2ca02c"),
            ),
            row=1,
            col=2,
        )

        fig.add_trace(
            go.Scatter(
                x=self.epochs,
                y=self.val_acc,
                mode="lines+markers",
                name="val_acc",
                line=dict(color="#ff7f0e"),
            ),
            row=1,
            col=2,
        )

        # Axis labels
        fig.update_xaxes(title_text="Epoch", row=1, col=1)
        fig.update_xaxes(title_text="Epoch", row=1, col=2)
        fig.update_yaxes(title_text="Loss", rangemode="tozero", row=1, col=1)
        fig.update_yaxes(title_text="Accuracy", rangemode="tozero", row=1, col=2)

        fig.update_layout(
            height=350,
            width=880,
            showlegend=True,
            template="plotly_white",
            margin=dict(l=30, r=30, t=50, b=30),
        )

        fig.show()

## 2. Carregamento e Exploração dos Dados

### Sobre o Fashion MNIST
- Estrutura (28x28 pixels, grayscale)
- 10 classes de roupas: [T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot]
- 60,000 imagens de treino e 10,000 de validação

In [3]:
# Keras já fornece o Fashion MNIST preprocessado e dividido
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# Mapeamento das labels para nomes legíveis
class_map = {
    0: "T-shirt/top",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot",
}

In [4]:
print(f"x_train.shape: {x_train.shape}")
print(f"y_train.shape: {y_train.shape}")
print(f"x_test.shape: {x_test.shape}")
print(f"y_test.shape: {y_test.shape}")
print(f"\nRange dos valores dos pixels: [{x_train.min()}, {x_train.max()}]")

x_train.shape: (60000, 28, 28)
y_train.shape: (60000,)
x_test.shape: (10000, 28, 28)
y_test.shape: (10000,)

Range dos valores dos pixels: [0, 255]


## 3. Visualização das Amostras

Mesmo do notebook [anterior](01_fashion_mnist_ml.ipynb)

## 4. Análise da Distribuição das Classes

Mesmo do notebook [anterior](01_fashion_mnist_ml.ipynb)

## 5. Pré-processamento para CNN

Diferença Crítica: Reshape vs Flatten

**Diferença Crítica: reshape na CNN vs flatten na Arquitetura Densa**
No notebook anterior (modelo Denso/MLP), as imagens eram achatadas em um vetor 1D, o que elimina completamente a estrutura espacial:

```python
x_train.reshape(-1, 784)  # (60000, 784) - vetor 1D
```

Esse formato funciona para MLPs, mas impede que o modelo aproveite relações locais entre pixels — essenciais em visão computacional.

**CNNs preservam a estrutura espacial**

Neste notebook, mantemos o formato original das imagens, adicionando apenas o eixo de canal:

```python
x_train.reshape(-1, 28, 28, 1)  # (60000, 28, 28, 1) - mantém 2D + canal
```

Assim, a CNN recebe as imagens no formato altura * largura * canais, possibilitando que filtros convolucionais explorem padrões locais como bordas, texturas e formas.

**Observação: o número de canais**

O último eixo indica quantos canais de cor a imagem possui:

1 -> imagens grayscale (como o Fashion-MNIST)

3 -> imagens RGB (Red, Green, Blue)

4 -> imagens com canal alfa (RGBA)

In [5]:
# Normalização
x_train_normalized = x_train.astype("float32") / 255.0
x_test_normalized = x_test.astype("float32") / 255.0

# Reshape CRUCIAL para CNNs: (samples, height, width, channels)
x_train_cnn = x_train_normalized.reshape(-1, 28, 28, 1)
x_test_cnn = x_test_normalized.reshape(-1, 28, 28, 1)

print(f"x_train_cnn: {x_train_cnn.shape}")

x_train_cnn: (60000, 28, 28, 1)


### Codificação Categórica dos Labels

**One-Hot Encoding:** Converter labels de inteiros (0-9) para vetores binários (similar ao  [LabelBinarizer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html) do scikit-learn)
- Exemplo: classe 2 -> [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
- Necessário para usar categorical_crossentropy como loss function
- Permite que o modelo retorne probabilidades para cada classe

In [6]:
# One-hot encoding dos labels
num_classes = len(np.unique(y_train))
num_samples = y_train.shape[0]

y_train_binary = keras.utils.to_categorical(y_train, num_classes)
y_test_binary = keras.utils.to_categorical(y_test, num_classes)

print(
    f"Label original de '{class_map[y_train[0]]}': {y_train[0]}, binarizado: {y_train_binary[0]}"
)

assert y_train_binary.shape == (
    num_samples,
    num_classes,
), "Houve um erro, o número de linhas ou colunas é inconsistente"

Label original de 'Ankle boot': 9, binarizado: [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


## 6. Construção da CNN - Arquitetura Detalhada

Design Pattern: Convolução -> Normalização -> Pooling

Este padrão se repete 3 vezes, cada vez:
- Aumentando a profundidade (número de filtros)
- Reduzindo dimensões espaciais (pooling)
- Extraindo features cada vez mais abstratas

In [7]:
model = Sequential(
    [
        Input(shape=(28, 28, 1)),
        # Extração Inicial de Features (Baixo Nível)
        # Conv2D Layer 1: Detecta bordas, linhas e texturas básicas
        # - 75 filtros de 3*3 -> aprende 75 padrões diferentes
        # - padding='same' mantém saída 28*28
        # - ReLU adiciona não-linearidade
        Conv2D(
            filters=75,
            kernel_size=(3, 3),
            strides=1,
            padding="same",
            activation="relu",
            input_shape=(28, 28, 1),
            name="conv2d_1",
        ),
        # BatchNormalization:
        # - Normaliza ativações
        # - Reduz oscilações internas (Internal Covariate Shift)
        # - Treinamento mais rápido e estável
        BatchNormalization(name="batch_norm_1"),
        # MaxPooling:
        # - Reduz dimensionalidade espacial (28*28 -> 14*14)
        # - Torna o modelo mais robusto a pequenas translações
        # - Reduz custo computacional
        MaxPool2D(pool_size=(2, 2), strides=2, padding="same", name="max_pool_1"),
        # Features Intermediárias (Padrões Combinados)
        # Conv2D Layer 2:
        # - Aprende combinações de features iniciais -> formas mais complexas
        # - Saída: (14, 14, 50)
        Conv2D(
            filters=50,
            kernel_size=(3, 3),
            strides=1,
            padding="same",
            activation="relu",
            name="conv2d_2",
        ),
        # Dropout:
        # - Desliga 20% dos neurônios
        # - Reduz overfitting e força o modelo a generalizar
        Dropout(rate=0.2, name="dropout_1"),
        BatchNormalization(name="batch_norm_2"),
        # MaxPooling: (14, 14) -> (7, 7)
        MaxPool2D(pool_size=(2, 2), strides=2, padding="same", name="max_pool_2"),
        # Conv2D Layer 3:
        # - Captura padrões mais abstratos
        # - Reconhece detalhes como gola, manga, bolso, textura etc.
        # - Saída: (7, 7, 25)
        Conv2D(
            filters=25,
            kernel_size=(3, 3),
            strides=1,
            padding="same",
            activation="relu",
            name="conv2d_3",
        ),
        BatchNormalization(name="batch_norm_3"),
        # MaxPooling final: (7, 7) -> (4, 4)
        MaxPool2D(pool_size=(2, 2), strides=2, padding="same", name="max_pool_3"),
        # Fully Connected Layers
        # Flatten:
        # - Converte feature maps 3D (4*4*25) em vetor 1D (400)
        # - Necessário antes de camadas densas
        Flatten(name="flatten"),
        # Dense Layer:
        # - 512 neurônios para combinar as features aprendidas
        # - ReLU continua trazendo não-linearidade
        Dense(units=512, activation="relu", name="dense_1"),
        # Dropout:
        # - Desliga 30% dos neurônios
        # - Reduz overfitting e força o modelo a generalizar
        Dropout(rate=0.3, name="dropout_2"),
        Dense(units=num_classes, activation="softmax", name="output", dtype="float32"),
    ],
    name="Fashion_MNIST_CNN",
)

/home/kappa/image-classification/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1763316908.843190  142802 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5562 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:06:00.0, compute capability: 8.6


## 7. Análise da Arquitetura do Modelo

### Calculando a quantidade de parâmetros:
- **Cálculo:** (3 * 3 * 1 * 75) + 75 + (3 * 3 * 75 * 50) + 50 + (3 * 3 * 50 * 25) + 25 + (400 * 512) + 512 + (512 * 10) + 10

- **Total de parâmetros:** ~260k parâmetros treináveis

In [8]:
model.summary()

Model: "Fashion_MNIST_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 75)     │           750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_1                    │ (None, 28, 28, 75)     │           300 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool_1 (MaxPooling2D)       │ (None, 14, 14, 75)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 50)     │        33,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 14, 14, 50)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_2                    │ (None, 14, 14, 50)     │           200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool_2 (MaxPooling2D)       │ (None, 7, 7, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 7, 7, 25)       │        11,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_norm_3                    │ (None, 7, 7, 25)       │           100 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pool_3 (MaxPooling2D)       │ (None, 4, 4, 25)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │       205,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,867 (1003.39 KB)

 Trainable params: 256,567 (1002.21 KB)

 Non-trainable params: 300 (1.17 KB)

## 8. Compilação do Modelo

**Loss Function (categorical_crossentropy):**
- Mede o quão "erradas" estão as predições
- Ideal para classificação multi-classe com one-hot encoding
- Fórmula: -Σ(y_true * log(y_pred))

**Optimizer (adam - implícito por padrão):**
- Adam = Adaptive Moment Estimation
- Ajusta a learning rate automaticamente
- Geralmente funciona bem sem tunning manual

**Metrics (accuracy):**
- % de predições corretas
- Fácil de interpretar e se cai bem já que nosso dataset é balanceado

In [9]:
model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

## 9. Treinamento do Modelo

### Hiperparâmetros Importantes:

**Epochs (20):**
- Em cada época o modelo vê todos os dados de treino uma vez.
- Aumentar o número de épocas implica mais tempo de aprendizado.
- E também aumenta o risco de overfitting.

**Batch Size (32):**
- O batch size é a quantidade de amostras que a rede neural processa antes de realizar uma atualização nos pesos via backpropagation.
- Batches maiores   
Aproveitam melhor o paralelismo da GPU, permitindo treinar mais rápido por época.
Produzem um gradiente mais estável, porém podem levar a um modelo com menor capacidade de generalização.
- Batches menores
Geram mais passos de atualização por época, tornando o treinamento mais lento.
No entanto, o gradiente mais “ruidoso” pode atuar como uma forma de regularização, resultando às vezes em melhor desempenho final.
- Influenciado pela quantidade de VRAM/RAM

**Validation Data:**
- Usado para monitorar performance em dados não vistos.
- Quando observamos que a val_accuracy é muito menor que o train_accuracy temos um grande indício de `overfitting`.

In [10]:
plot_callback = PlotLearning()

train_start = time.time()

history = model.fit(
    x_train_cnn,
    y_train_binary,
    epochs=20,
    batch_size=32,
    verbose=1,
    validation_data=(x_test_cnn, y_test_binary),
    callbacks=[plot_callback],
)

train_end = time.time()

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9664 - loss: 0.0889 - val_accuracy: 0.9233 - val_loss: 0.2664


## 10. Análise dos Resultados de Treinamento

Pontos a serem obersvados:

- Training accuracy crescendo significa que o modelo está aprendendo
- Validation accuracy ficou estagnado, significa que apesar de performando bem no test, na validação não está, indicando um possível overfitting
- Quanto maior o `gap` entre train e val, mais overfitting
- O Validation loss aumentando é um sinal claro de overfitting

## 11. Comparação Direta: Dense vs CNN

Vamos comparar as duas abordagens lado a lado

In [20]:
test_loss, test_accuracy = model.evaluate(x_test_cnn, y_test_binary, verbose=0)

inf_start = time.time()
model.predict(
    x_test_cnn[[0]]
)  # Deveriamos rodar multiplas vezes para ter uma estimativa mais real
inf_end = time.time()

print(f"Total Training time: {train_end - train_start:.2f}s")
print(f"Inference time: {(inf_end - inf_start)*1000:.4f}ms/sample")

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Total Training time: 140.64s
Inference time: 50.1161ms/sample
Test Loss: 0.2664
Test Accuracy: 0.9233 (92.33%)


## 12. Matriz de Confusão - CNN

In [21]:
import numpy as np
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

y_pred = model.predict(x_test_cnn)
y_pred_classes = np.argmax(y_pred, axis=1)

# Gerar matriz de confusão
cm = confusion_matrix(y_test, y_pred_classes)

# Nomes das classes
labels = list(class_map.values())

# Criar heatmap
fig = go.Figure(
    data=go.Heatmap(
        z=cm,
        x=labels,
        y=labels,
        colorscale="Blues",
        colorbar=dict(title="Número de Predições"),
        text=cm,
        texttemplate="%{text}",
        showscale=True,
    )
)

# Layout
fig.update_layout(
    title="Matriz de Confusão",
    xaxis=dict(title="Classe Predita", tickangle=30),
    yaxis=dict(title="Classe Real", autorange="reversed"),
    width=700,
    height=600,
    margin=dict(l=60, r=40, t=70, b=60),
)

fig.show()

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


## 13. Relatório de Classificação Detalhado

**Precision:** Das predições positivas, quantas estavam corretas?

**Recall:** Dos casos positivos reais, quantos foram detectados?

**F1-Score:** Média harmônica entre precision e recall

**Support:** Número de amostras de cada classe no teste

In [22]:
print(
    classification_report(
        y_test, y_pred_classes, target_names=class_map.values(), digits=4
    )
)

              precision    recall  f1-score   support

 T-shirt/top     0.9187    0.8140    0.8632      1000
     Trouser     0.9861    0.9920    0.9890      1000
    Pullover     0.9276    0.8460    0.8849      1000
       Dress     0.9265    0.9330    0.9297      1000
        Coat     0.8361    0.9230    0.8774      1000
      Sandal     0.9909    0.9850    0.9880      1000
       Shirt     0.7461    0.8110    0.7772      1000
     Sneaker     0.9653    0.9740    0.9696      1000
         Bag     0.9880    0.9850    0.9865      1000
  Ankle boot     0.9719    0.9700    0.9710      1000

    accuracy                         0.9233     10000
   macro avg     0.9257    0.9233    0.9237     10000
weighted avg     0.9257    0.9233    0.9237     10000



## 14. Visualização de Features Aprendidas

Vamos visualizar os filtros da primeira camada convolucional
para entender o que a rede está aprendendo

In [23]:
# Selecionar exemplos aleatórios
num_examples = 20

np.random.seed(42)
random_indices = np.random.choice(len(x_test), num_examples, replace=False)

fig = make_subplots(
    rows=4,
    cols=5,
    subplot_titles=[
        f"Real: {class_map[y_test[idx]]}<br>Pred: {class_map[y_pred_classes[idx]]}<br>({np.max(y_pred[idx])*100:.1f}%)"
        for idx in random_indices
    ],
    horizontal_spacing=0.02,
    vertical_spacing=0.08,
)

for i, idx in enumerate(random_indices):
    row = (i // 5) + 1
    col = (i % 5) + 1

    fig.add_trace(
        go.Heatmap(
            z=np.flipud(x_test[idx]),  # Inverte verticalmente
            colorscale="gray",
            showscale=False,
        ),
        row=row,
        col=col,
    )

    color = "green" if y_test[idx] == y_pred_classes[idx] else "red"
    fig.layout.annotations[i].update(font=dict(color=color, size=9))

fig.update_xaxes(showticklabels=False, matches=None)
fig.update_yaxes(showticklabels=False, matches=None, scaleanchor="x", scaleratio=1)
fig.update_layout(
    height=800,
    width=800,
    showlegend=False,
    title_text="Exemplos de Predições (Verde=Correto, Vermelho=Erro)",
    title_font_size=14,
    margin=dict(l=10, r=10, t=80, b=10),
)

fig.show()

## 15. Análise dos Erros Mais Confiantes

Erros com alta confiança indicam confusões sistemáticas no modelo

In [24]:
# Encontrar erros com alta confiança
errors_mask = y_test != y_pred_classes
confident_errors = np.where(errors_mask & (np.max(y_pred, axis=1) > 0.9))[0]

if len(confident_errors) > 0:
    print(f"Encontrados {len(confident_errors)} erros com confiança > 90%\n")

    # Mostrar até 10 exemplos
    num_to_show = min(10, len(confident_errors))

    fig = make_subplots(
        rows=2,
        cols=5,
        subplot_titles=[
            f"Real: {class_map[y_test[idx]]}<br>Pred: {class_map[y_pred_classes[idx]]}<br>({np.max(y_pred[idx])*100:.1f}%)"
            for idx in confident_errors[:num_to_show]
        ],
        horizontal_spacing=0.02,
        vertical_spacing=0.15,
    )

    for i, idx in enumerate(confident_errors[:num_to_show]):
        row = (i // 5) + 1
        col = (i % 5) + 1

        fig.add_trace(
            go.Heatmap(z=np.flipud(x_test[idx]), colorscale="gray", showscale=False),
            row=row,
            col=col,
        )

        fig.layout.annotations[i].update(font=dict(color="darkred", size=9))

    fig.update_xaxes(showticklabels=False, matches=None)
    fig.update_yaxes(showticklabels=False, matches=None, scaleanchor="x", scaleratio=1)
    fig.update_layout(
        height=400,
        width=800,
        showlegend=False,
        title_text="Erros Mais confiantes (>90% confiança)",
        title_font_size=14,
        margin=dict(l=10, r=10, t=80, b=10),
    )

    fig.show()
else:
    print("Nenhum erro com confiança > 90% encontrado!")

Encontrados 274 erros com confiança > 90%



## 16. Conclusões e Insights Arquiteturais

### Performance Comparativa

In [30]:
comparison_data = {
    "Métrica": [
        "Arquitetura",
        "Parâmetros",
        "Estrutura dos Dados",
        "Informação Espacial",
        "Test Accuracy",
        "Overfitting",
        "Tempo de total de Treino",
        "Tempo de inferência",
    ],
    "Dense (Notebook 1)": [
        "Fully Connected",
        "~670,000",
        "Flatten (784,)",
        "Perdida",
        "~89.06%",
        "Alto",
        "~114 s",
        "~256 ms/sample",
    ],
    "CNN (Este Notebook)": [
        "Convolutional",
        "~250,000",
        "2D (28, 28, 1)",
        "Preservada",
        f"{test_accuracy*100:.2f}%",
        "Baixo",
        "~140 s",
        "~50 ms/sample",
    ],
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

                 Métrica Dense (Notebook 1) CNN (Este Notebook)
             Arquitetura    Fully Connected       Convolutional
              Parâmetros           ~670,000            ~250,000
     Estrutura dos Dados     Flatten (784,)      2D (28, 28, 1)
     Informação Espacial            Perdida          Preservada
           Test Accuracy            ~89.06%              92.33%
             Overfitting               Alto               Baixo
Tempo de total de Treino             ~114 s              ~140 s
     Tempo de inferência     ~256 ms/sample       ~50 ms/sample
